# 🛠️ GIAI ĐOẠN 1: DATA PROFILING & DATA CLEANING
## HỆ THỐNG BÁO CÁO TAI NẠN GIAO THÔNG (TRAFFIC ACCIDENT DWH)

### 📌 Bối cảnh & Mục tiêu (Context & Objective)
* **Mục tiêu dự án:** Chuyển đổi và nâng cấp hệ thống kho dữ liệu cũ (Legacy BI Stack: SSIS + SSAS) sang Modern Analytics Engineering Stack (PostgreSQL + dbt + Power BI).
* **Nhiệm vụ file 01:** Thực hiện thám hiểm cấu trúc hình thái dữ liệu (Data Profiling) trên tập dữ liệu thô (~200K records) để phát hiện các lỗi hệ thống (Null, Duplicate, Sai kiểu dữ liệu). Từ đó, xây dựng đường ống làm sạch (Data Cleaning pipeline) nhằm cung cấp một tập dữ liệu phẳng, tinh khiết phục vụ cho giai đoạn EDA và xây dựng Star Schema.
* **Nguyên tắc cốt lõi:** Đảm bảo giữ vững tính toàn vẹn dữ liệu (Data Integrity) để phục vụ bước **Migration Validation (Đối soát khớp 100% kết quả với hệ thống cũ)**.

## 🔍 1. DATA PROFILING (Thám hiểm & Đánh giá sức khỏe dữ liệu)

### 📊 1.1 Kiểm tra hình thái & Cấu trúc tổng quan
* **Mục tiêu:** Xác định số lượng dòng/cột thực tế, nhận diện các trường khóa chính/khóa ngoại tiềm năng, và kiểm tra mức độ trống (Missing Values) trên toàn bộ các thuộc tính.
* **Kế hoạch hành động:** Sử dụng các hàm hệ thống để quét nhanh phân phối kiểu dữ liệu (Data Types) hiện tại của tệp CSV nguồn.

In [ ]:
# Import thư viện xử lý dữ liệu
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import thư viện vẽ biểu đồ
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')
sns.set_palette('husl')

# Import SMOTE để cân bằng dữ liệu
from imblearn.over_sampling import SMOTE

print("Đã import thành công tất cả thư viện cần thiết.")

Đã import thành công tất cả thư viện cần thiết.


### Đọc dữ liệu từ file CSV

In [ ]:
# Đọc dữ liệu
df = pd.read_csv('traffic_accidents.csv')

print(f"Kích thước dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột")

Kích thước dữ liệu: 209,306 dòng x 24 cột


### Xem thông tin tổng quan về dữ liệu

In [ ]:
# Hiển thị 5 dòng đầu tiên
print("\n5 DÒNG ĐẦU TIÊN CỦA DỮ LIỆU:")
df.head()


5 DÒNG ĐẦU TIÊN CỦA DỮ LIỆU:


,crash_date,traffic_control_device,weather_condition,lighting_condition,first_crash_type,trafficway_type,alignment,roadway_surface_cond,road_defect,crash_type,...,most_severe_injury,injuries_total,injuries_fatal,injuries_incapacitating,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,crash_hour,crash_day_of_week,crash_month
0,07/29/2023 01:00:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,TURNING,NOT DIVIDED,STRAIGHT AND LEVEL,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,13,7,7
1,08/13/2023 12:11:00 AM,TRAFFIC SIGNAL,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,2.0,0,1,8
2,12/09/2021 10:30:00 AM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,10,5,12
3,08/09/2023 07:55:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,ANGLE,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,...,NONINCAPACITATING INJURY,5.0,0.0,0.0,5.0,0.0,0.0,19,4,8
4,08/19/2023 02:55:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,14,7,8


In [ ]:
# Thông tin về các cột
print("\n THÔNG TIN CÁC CỘT:")
df.info()


 THÔNG TIN CÁC CỘT:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209306 entries, 0 to 209305
Data columns (total 24 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   crash_date                     209306 non-null  object 
 1   traffic_control_device         209306 non-null  object 
 2   weather_condition              209306 non-null  object 
 3   lighting_condition             209306 non-null  object 
 4   first_crash_type               209306 non-null  object 
 5   trafficway_type                209306 non-null  object 
 6   alignment                      209306 non-null  object 
 7   roadway_surface_cond           209306 non-null  object 
 8   road_defect                    209306 non-null  object 
 9   crash_type                     209306 non-null  object 
 10  intersection_related_i         209306 non-null  object 
 11  damage                         209306 non-null  object 
 12  prim_cont

In [ ]:
# Thống kê mô tả cho các cột số
print("\n THỐNG KÊ MÔ TẢ:")
df.describe()


 THỐNG KÊ MÔ TẢ:


,num_units,injuries_total,injuries_fatal,injuries_incapacitating,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,crash_hour,crash_day_of_week,crash_month
count,209306.000000,209306.000000,209306.000000,209306.000000,209306.000000,209306.000000,209306.000000,209306.000000,209306.000000,209306.000000
mean,2.063300,0.382717,0.001859,0.038102,0.221241,0.121516,2.244002,13.373047,4.144024,6.771822
std,0.396012,0.799720,0.047502,0.233964,0.614960,0.450865,1.241175,5.603830,1.966864,3.427593
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
25%,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,9.000000,2.000000,4.000000
50%,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,14.000000,4.000000,7.000000
75%,2.000000,1.000000,0.000000,0.000000,0.000000,0.000000,3.000000,17.000000,6.000000,10.000000
max,11.000000,21.000000,3.000000,7.000000,21.000000,15.000000,49.000000,23.000000,7.000000,12.000000


### Cardinality của các Dimension

In [ ]:
def profile_cardinality(df, categorical_cols):
    """
    Trả về DataFrame hiển thị số lượng giá trị duy nhất (unique values)
    và danh sách các giá trị đó để đánh giá cấu trúc Dimension.
    """
    cardinality_data = []
    for col in categorical_cols:
        unique_cnt = df[col].nunique()
        sample_vals = df[col].unique()[:5].tolist()
        cardinality_data.append({
            'Column': col,
            'Unique_Count': unique_cnt,
            'Samples': sample_vals
        })

    result_df = pd.DataFrame(cardinality_data)

    if result_df.empty:
        print("Không tìm thấy cột dữ liệu dạng chuỗi (string/object).")
        return result_df
    return result_df.sort_values(by='Unique_Count', ascending=False)

categorical_columns = df.select_dtypes(include=['object']).columns
profile_cardinality(df, categorical_columns)

,Column,Unique_Count,Samples
0,crash_date,189087,"[07/29/2023 01:00:00 PM, 08/13/2023 12:11:00 A..."
12,prim_contributory_cause,40,"[UNABLE TO DETERMINE, IMPROPER TURNING/NO SIGN..."
5,trafficway_type,20,"[NOT DIVIDED, FOUR WAY, T-INTERSECTION, DIVIDE..."
1,traffic_control_device,19,"[TRAFFIC SIGNAL, NO CONTROLS, STOP SIGN/FLASHE..."
4,first_crash_type,18,"[TURNING, REAR END, ANGLE, FIXED OBJECT, REAR ..."
2,weather_condition,12,"[CLEAR, RAIN, SNOW, CLOUDY/OVERCAST, UNKNOWN]"
8,road_defect,7,"[UNKNOWN, NO DEFECTS, OTHER, SHOULDER DEFECT, ..."
7,roadway_surface_cond,7,"[UNKNOWN, DRY, WET, SNOW OR SLUSH, ICE]"
3,lighting_condition,6,"[DAYLIGHT, DARKNESS, LIGHTED ROAD, DUSK, DARKN..."
6,alignment,6,"[STRAIGHT AND LEVEL, CURVE, LEVEL, STRAIGHT ON..."


### Kiểm tra Outliers

In [ ]:
def detect_numerical_outliers(df, numerical_cols):
    """
    Tính toán số lượng dòng bị coi là Outlier của từng cột số dựa trên IQR
    để phát hiện các dữ liệu bất thường (Anomalies).
    """
    outlier_report = {}
    for col in numerical_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_report[col] = {
            'Lower_Bound': lower_bound,
            'Upper_Bound': upper_bound,
            'Outlier_Count': outliers.shape[0],
            'Percentage (%)': round((outliers.shape[0] / df.shape[0]) * 100, 2)
        }
    return pd.DataFrame(outlier_report).T

num_cols = ['num_units', 'injuries_total', 'injuries_fatal', 'injuries_incapacitating']
detect_numerical_outliers(df, num_cols)

,Lower_Bound,Upper_Bound,Outlier_Count,Percentage (%)
num_units,2.0,2.0,19940.0,9.53
injuries_total,-1.5,2.5,5692.0,2.72
injuries_fatal,0.0,0.0,351.0,0.17
injuries_incapacitating,0.0,0.0,6634.0,3.17


### 📝 KẾT LUẬN VÀ PHÂN TÍCH CHẤT LƯỢNG DỮ LIỆU (DATA PROFILING INSIGHTS)

Sau khi thực hiện phân tích chất lượng dữ liệu dựa trên đặc tính số lượng giá trị phân biệt (Cardinality) và dữ liệu dị biệt (Outlier), hệ thống ghi nhận các kết luận sau để định hình cho tầng tiền xử lý dữ liệu và thiết kế mô hình:

#### 1. Về cấu trúc mô hình dữ liệu (Data Modeling Insights)
* **Xác thực kiến trúc Star Schema:** Ngoại trừ thuộc tính thời gian (`crash_date`), toàn bộ các cột định tính (Categorical Columns) như `weather_condition`, `lighting_condition`, `roadway_surface_cond`, `prim_contributory_cause` đều có số lượng giá trị duy nhất rất thấp (từ 2 đến 40 giá trị). Kết quả này chứng minh việc tách các thuộc tính này thành các bảng chiều độc lập (Dimension Tables) là tối ưu, giúp giảm dung lượng lưu trữ và tăng tốc độ truy vấn.
* **Độ mịn (Granularity) của trường thời gian:** Thuộc tính `crash_date` chứa đến 189,087 giá trị phân biệt do ghi nhận chi tiết đến từng giờ-phút-giây. Để tối ưu hiệu năng, tại tầng biến đổi dữ liệu (ETL/dbt), hệ thống bắt buộc phải phân rã: trích xuất phần ngày để tạo bảng chiều thời gian (`dim_date`) và đẩy phần giờ (`crash_hour`) về bảng sự kiện (`fact_accident`).
* **Gom nhóm bảng chiều:** Các thuộc tính có chung ngữ cảnh môi trường tự nhiên như thời tiết (`weather_condition`), ánh sáng (`lighting_condition`), và bề mặt đường (`roadway_surface_cond`) sẽ được gom chung vào một bảng chiều duy nhất mang tên `dim_natural_condition` nhằm giảm số lượng phép toán kết hợp (`JOIN`) khi vận hành báo cáo.

#### 2. Về chất lượng và quy tắc nghiệp vụ (Data Quality & Business Rules)
* **Đặc trưng phân phối dữ liệu:** Thuật toán IQR xác định các vụ tai nạn có số lượng phương tiện lớn (`num_units > 2`) hoặc có thương vong (`injuries_fatal > 0`, `injuries_incapacitating > 0`) là dữ liệu dị biệt (Outliers). Điều này phù hợp thực tế vì phần lớn các vụ va chạm giao thông đô thị chỉ gồm 2 xe va quệt nhẹ và không có thiệt hại về người.
* **Nhật ký lỗi chất lượng dữ liệu (Data Quality Issues Log):** Quá trình Profiling đã phát hiện 4 vấn đề nghiêm trọng cần xử lý:
    1. *Sai lệch kiểu dữ liệu (Data Type Mismatch):* Cột `crash_date` đang ở dạng chuỗi (Object/String). Các cột đếm số lượng chấn thương đang bị định dạng thành kiểu số thực (`float64`) do chứa ô trống.
    2. *Trùng lặp dữ liệu (Data Duplication):* Xuất hiện các bản ghi trùng lặp hoàn toàn trên tất cả các thuộc tính định danh, gây sai số cho các phép toán gộp (`COUNT`, `SUM`).
    3. *Tỷ lệ dữ liệu khuyết thiếu mang nhãn 'UNKNOWN' cao:* Cột `road_defect` chứa giá trị ẩn `'UNKNOWN'` lên tới 16.45%, không đủ độ tin cậy để làm thuộc tính phân tích. Các trường môi trường còn lại có tỷ lệ khuyết thiếu thấp dưới 5%.
    4. *Tiêu chí đánh giá mức độ nghiêm trọng chưa rõ ràng:* Biến mục tiêu đo lường mức độ nghiêm trọng (`is_severe`) chưa được định nghĩa chặt chẽ theo chuẩn nghiệp vụ mới.

---

## 🛠️ 2. DATA PREPROCESSING & CLEANING PIPELINE (Đường ống tiền xử lý dữ liệu)

### 📐 2.1 Chiến lược xử lý và Quy tắc chuyển đổi dữ liệu (Transformation Rules)
Để giải quyết triệt để 4 vấn đề chất lượng dữ liệu được phát hiện ở giai đoạn Profiling mà không làm mất tính toàn vẹn dữ liệu phục vụ đối soát (Migration Validation), hệ thống áp dụng các quy tắc chuyển đổi sau:

* **Chuẩn hóa cấu trúc cột và Kiểu dữ liệu (Schema Alignment):** * Chuyển đổi chuỗi ngày tháng thô `crash_date` sang định dạng chuẩn `datetime64[ns]`.
    * Ép lại toàn bộ các cột định tính (Categorical fields) sang kiểu dữ liệu chuỗi kí tự (`string`) chuyên dụng của Pandas để tối ưu hóa bộ nhớ xử lý.
    * Thêm một cột year để chuẩn bị cho bước thiết lập hierachy sau này
* **Loại bỏ trùng lặp và Lọc dòng lỗi (Data Filtering):**
    * Loại bỏ hoàn toàn các dòng trùng lặp vật lý (`drop_duplicates`) để làm sạch dữ liệu đầu vào.
    * Loại bỏ hoàn toàn cột thuộc tính `road_defect` do không vượt qua bài kiểm tra mật độ dữ liệu chất lượng (`UNKNOWN` > 16%).
    * Thực hiện loại bỏ các dòng chứa giá trị `'UNKNOWN'` tại các cột thuộc tính có tỷ lệ khuyết nhỏ (`< 5%`) bao gồm `lighting_condition`, `traffic_control_device`, và `trafficway_type` để đảm bảo độ tinh khiết cho các bảng chiều sau này.
* **Tái định nghĩa biến mục tiêu (`is_severe`):** Áp dụng quy tắc nghiệp vụ mới một cách nghiêm ngặt. Hệ thống chỉ gán cờ `is_severe = 1` (Nghiêm trọng) khi vụ tai nạn ghi nhận có ca tử vong (`injuries_fatal > 0`) hoặc chấn thương mất năng lực hành vi (`injuries_incapacitating > 0`). Các trường hợp còn lại gán cờ `0`.

> **ĐÚC KẾT CỐT LÕI (Key Takeaway):** Toàn bộ quy trình tiền xử lý dữ liệu dưới đây tuân thủ nguyên tắc loại bỏ các thuộc tính nhiễu cấu trúc (như `road_defect` và dữ liệu trùng lặp) nhưng bảo toàn tối đa số lượng dòng thô hợp lệ, thiết lập một nền tảng dữ liệu sạch chuẩn hóa trước khi thực hiện phân tích chuyên sâu và phân rã mô hình Star Schema.

In [ ]:
# Tạo biến target: is_severe
df['is_severe'] = ((df['injuries_fatal'] > 0) | (df['injuries_incapacitating'] > 0)).astype(int)

# Kiểm tra phân phối
print("\nPHÂN PHỐI BIẾN MỤC TIÊU:")
print(df['is_severe'].value_counts())
print(f"\nTỷ lệ:")
print(df['is_severe'].value_counts(normalize=True) * 100)


PHÂN PHỐI BIẾN MỤC TIÊU:
is_severe
0    202391
1      6915
Name: count, dtype: int64

Tỷ lệ:
is_severe
0    96.696225
1     3.303775
Name: proportion, dtype: float64


### Xử lý giá trị thiếu

In [ ]:
# Kiểm tra giá trị thiếu
print("\nKIỂM TRA GIÁ TRỊ THIẾU:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])


KIỂM TRA GIÁ TRỊ THIẾU:
Series([], dtype: int64)


In [ ]:
df = df.dropna()
print(f"\n Dữ liệu sau khi xử lý: {df.shape}")


 Dữ liệu sau khi xử lý: (209306, 25)


In [ ]:
# Kiểm tra giá trị bị trùng
print("\n KIỂM TRA GIÁ TRỊ TRÙNG:")
print(df.duplicated().sum())

# Loại bỏ các dòng có giá trị trùng (nếu có)
df = df.drop_duplicates()

print(f"\n Dữ liệu sau khi xử lý: {df.shape}")


 KIỂM TRA GIÁ TRỊ TRÙNG:
31

 Dữ liệu sau khi xử lý: (209275, 25)


In [ ]:
def check_missing_ratio(df, target_value='UNKNOWN'):
    """
    Hàm tính tỉ lệ phần trăm của một giá trị cụ thể trong các cột của DataFrame.
    """
    # Tính tỉ lệ % cho toàn bộ các cột
    ratios = (df == target_value).mean() * 100

    # Chỉ lọc ra các cột có chứa giá trị này và sắp xếp giảm dần
    ratios = ratios[ratios > 0].sort_values(ascending=False)

    # Tạo DataFrame để hiển thị đẹp hơn
    result_df = pd.DataFrame({
        'Số lượng': (df == target_value).sum()[ratios.index],
        'Tỉ lệ (%)': ratios.round(2) # Làm tròn 2 chữ số thập phân
    })

    return result_df

# --- Áp dụng vào dữ liệu của bạn ---
print(f"\nKIỂM TRA TỈ LỆ GIÁ TRỊ '{'UNKNOWN'}':")
unknown_report = check_missing_ratio(df, target_value='UNKNOWN')
print(unknown_report)


KIỂM TRA TỈ LỆ GIÁ TRỊ 'UNKNOWN':
                        Số lượng  Tỉ lệ (%)
road_defect                34423      16.45
roadway_surface_cond       12509       5.98
weather_condition           6534       3.12
traffic_control_device      4455       2.13
lighting_condition          4336       2.07
trafficway_type             1060       0.51


In [ ]:
# Loại bỏ cột road_defect do tỉ lệ dữ liệu UNKNOWN cao nhất
df = df.drop(columns=['road_defect'])

# In ra số lượng cột còn lại để xác nhận
print(f"Kích thước dữ liệu sau khi loại bỏ cột: {df.shape}")

Kích thước dữ liệu sau khi loại bỏ cột: (209275, 24)


In [ ]:
def remove_unknown_rows(df, columns_to_clean, target_value='UNKNOWN'):
    """
    Hàm loại bỏ các dòng chứa giá trị được chỉ định (mặc định là 'UNKNOWN')
    trong danh sách các cột cụ thể.
    """
    # Ghi nhận số dòng ban đầu
    initial_rows = df.shape[0]

    # Lọc giữ lại các dòng KHÔNG chứa target_value
    for col in columns_to_clean:
        df = df[df[col] != target_value]

    # Tính toán số lượng đã xóa
    final_rows = df.shape[0]
    rows_dropped = initial_rows - final_rows

    print(f"Đã loại bỏ {rows_dropped} dòng chứa '{target_value}'.")
    print(f"Kích thước dữ liệu hiện tại: {df.shape}")

    return df

# --- Loại bỏ các dòng UNKNOWN của các cột thuộc tính này vì tỉ lệ UNKNOWN thấp (< 3~5%)---
cols_to_drop_rows = ['lighting_condition', 'traffic_control_device', 'trafficway_type']

df = remove_unknown_rows(df, columns_to_clean=cols_to_drop_rows)

Đã loại bỏ 8007 dòng chứa 'UNKNOWN'.
Kích thước dữ liệu hiện tại: (201268, 24)


### Thêm cột year

In [ ]:
df['crash_date'] = pd.to_datetime(df['crash_date'], format='%m/%d/%Y %I:%M:%S %p', errors= "coerce")

df['crash_year'] = df['crash_date'].dt.year

print(df[['crash_date','crash_year']].head())

           crash_date  crash_year
0 2023-07-29 13:00:00        2023
1 2023-08-13 00:11:00        2023
2 2021-12-09 10:30:00        2021
3 2023-08-09 19:55:00        2023
4 2023-08-19 14:55:00        2023


### Chuyển cột text sang string

In [ ]:
df = df.astype({col: "string" for col in df.select_dtypes(include="object").columns})
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 201268 entries, 0 to 209305
Data columns (total 25 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   crash_date                     201268 non-null  datetime64[ns]
 1   traffic_control_device         201268 non-null  string        
 2   weather_condition              201268 non-null  string        
 3   lighting_condition             201268 non-null  string        
 4   first_crash_type               201268 non-null  string        
 5   trafficway_type                201268 non-null  string        
 6   alignment                      201268 non-null  string        
 7   roadway_surface_cond           201268 non-null  string        
 8   crash_type                     201268 non-null  string        
 9   intersection_related_i         201268 non-null  string        
 10  damage                         201268 non-null  string        
 11  prim_

In [ ]:
# Lưu dữ liệu đã làm sạch để dùng cho các bước sau
df.to_csv("traffic_clean.csv", index=False)
print("Đã lưu dữ liệu sạch thành công!")

Đã lưu dữ liệu sạch thành công!
